# Age-Specific Death Rates and the Mortality Indicator Zoo

*Demographic Analysis Series · Module 1 (Mortality) · Lesson 1.1c*
*Author: Jason Li | Published: August 30, 2026*

---

[Lesson 1.1b](demographic-m1l1b-crude-death-rate) ended inside the master identity: a crude rate is a schedule $m(s,x)$ times a composition $p(s,x)$. This post gives the schedule its name, learns to read its shape, and then tours the indicators demographers built for the ages where mortality matters most: the infant mortality rate (IMR), the under-five mortality rate (U5MR), and the maternal mortality ratio (MMR). Each has a definition that looks obvious and a denominator that is not.

**What you'll learn**

- The age-specific death rate (ASDR): definition, notation, and why we always plot it on a log scale
- The J shape of mortality: infancy, the quiet ages, the accident hump, and the near-geometric rise after 30
- Why IMR and U5MR are probabilities, not rates, and why MMR is a ratio, not a rate
- Three denominator traps: the misnamed "rate", the period mismatch, and the exposure problem inside age 0

**Anchors:** Wachter §2.4; Carmichael Ch. 1 (measure types, the IMR worked example on p. 30).


---

## Part 1: The Age-Specific Death Rate and Its Curve

The refinement move from Lesson 1.1b, written formally, is the **age-specific death rate**:

$$
{}_nM_x = \frac{\text{deaths at ages } x \text{ to } x+n \text{ in a period}}{\text{mid-period population aged } x \text{ to } x+n} \times 1000
$$

With sex added, Carmichael writes it $\text{ASSDR}(s,x) = D(s,x)/P(s,x) \times 1000$. It is a *true* rate in his taxonomy: the denominator is the population genuinely at risk, averaged over the period. We reload the 1988 Malaysia and Australia schedules from Lesson 1.1b and look at them properly this time.


In [ ]:
# ---- 1. The mortality curve, linear scale vs log scale ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 12

# Table 1.2 (rates per 1,000) and Table 1.3 (persons per 1,000,000), Carmichael Ch. 1
rates_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,16.6,9.8,12.8,7.6
1-4,1.7,0.5,1.1,0.4
5-9,0.6,0.2,0.5,0.2
10-14,0.6,0.3,0.4,0.2
15-19,1.0,1.1,0.5,0.4
20-24,1.5,1.6,0.6,0.5
25-29,1.7,1.5,0.9,0.5
30-34,2.0,1.4,1.1,0.6
35-39,2.4,1.5,1.5,0.8
40-44,3.4,2.2,2.3,1.2
45-49,5.5,3.4,3.2,2.1
50-54,9.9,6.0,5.9,3.4
55-59,15.4,10.0,9.9,5.5
60-64,25.5,17.3,17.0,8.7
65-69,38.3,27.2,27.8,13.8
70-74,62.7,45.3,46.6,23.5
75-79,86.2,71.9,69.8,40.7
80-84,144.3,110.7,119.0,71.4
85+,173.4,186.6,142.8,147.7
"""
pop_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,14254,7597,13464,7245
1-4,56773,30324,53804,28954
5-9,61207,37762,58012,35803
10-14,56002,38712,53613,36758
15-19,52776,43626,50772,41743
20-24,50193,40771,49060,39297
25-29,42748,42914,44839,41982
30-34,35704,40234,38326,39933
35-39,30427,38778,31269,38391
40-44,23858,36248,23075,34537
45-49,20473,27923,19767,26357
50-54,17052,23855,17190,22798
55-59,12615,22708,13604,21924
60-64,10043,21764,10513,22336
65-69,7373,17701,8498,19973
70-74,4608,12857,5246,16200
75-79,3430,8712,3976,12555
80-84,1399,4471,1675,7827
85+,963,2319,1399,6111
"""
rates = pd.read_csv(StringIO(rates_csv), index_col='age')
pop = pd.read_csv(StringIO(pop_csv), index_col='age')
deaths = pop * rates / 1000

# both-sexes ASDR per country: total deaths / total persons in the age group
asdr = pd.DataFrame({
    'Malaysia': (deaths['m_malaysia'] + deaths['f_malaysia']) / (pop['m_malaysia'] + pop['f_malaysia']) * 1000,
    'Australia': (deaths['m_australia'] + deaths['f_australia']) / (pop['m_australia'] + pop['f_australia']) * 1000,
})

x = np.arange(len(asdr))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(x, asdr['Malaysia'], 'o-', color='#e0826d', lw=2, label='Malaysia')
axes[0].plot(x, asdr['Australia'], 's-', color='#6d8ee0', lw=2, label='Australia')
axes[0].set_title('(a) Linear scale')
axes[0].set_ylabel('deaths per 1,000')
for ax, scale in [(axes[1], 'log')]:
    ax.semilogy(x, asdr['Malaysia'], 'o-', color='#e0826d', lw=2, label='Malaysia')
    ax.semilogy(x, asdr['Australia'], 's-', color='#6d8ee0', lw=2, label='Australia')
    ax.set_title('(b) Log scale')
axes[1].set_ylabel('deaths per 1,000 (log)')
for ax in axes:
    ax.set_xticks(x[::2])
    ax.set_xticklabels(asdr.index[::2], rotation=45)
    ax.set_xlabel('age group')
    ax.legend()
plt.tight_layout()
plt.show()

# checkpoints: the anatomy of the J
imin = asdr['Malaysia'].idxmin()
print(f'Minimum of the curve: age {imin} (Malaysia {asdr.loc[imin, "Malaysia"]:.2f}, '
      f'Australia {asdr.loc[imin, "Australia"]:.2f} per 1,000)')
print(f'Range of the curve, Malaysia: {asdr["Malaysia"].max() / asdr["Malaysia"].min():.0f}x '
      f'from the safest age to 85+')
# near-geometric rise: mean 5-year ratio between ages 35-39 and 80-84
seg = asdr.loc['35-39':'80-84']
for c in asdr.columns:
    r = (seg[c].iloc[-1] / seg[c].iloc[0]) ** (1 / (len(seg) - 1))
    print(f'{c}: mean 5-year growth factor of ASDR (35-39 to 80-84) = {r:.2f}, '
          f'i.e. doubling every {5 * np.log(2) / np.log(r):.1f} years')
# the accident hump, young males (sex-specific, from Lesson 1.1b)
print('\nAccident hump check, males: at 15-19 and 20-24, '
      f'Australia ({rates.loc["15-19","m_australia"]}, {rates.loc["20-24","m_australia"]}) '
      f'>= Malaysia ({rates.loc["15-19","m_malaysia"]}, {rates.loc["20-24","m_malaysia"]}) per 1,000')

The two panels carry the same numbers, but they tell different stories.

On the **linear scale** the curve is an old-age story: everything before 50 is crushed against the axis, because mortality at 85+ (around 150 to 190 per 1,000) is hundreds of times mortality at 10–14 (0.2 to 0.5). On the **log scale**, equal *ratios* become equal *distances*, and the whole anatomy appears at once. Read it as four acts:

1. **Infancy**: mortality starts high at age 0 (Malaysia 14.8, Australia 8.7 per 1,000 both sexes) and falls steeply. This is why ages 0 and 1–4 get their own groups, and their own indicators, in Part 2.
2. **The quiet ages**: the minimum sits around 10–14, the safest ages of life in any population ever recorded.
3. **The accident hump**: rates rise again through 15–24, driven by external causes, especially among young men. Here Australian males (1.1, 1.6) sit slightly *above* Malaysian males (1.0, 1.5), the road-accident exception we met in Lesson 1.1b.
4. **Senescence**: from about 35 the log curve is nearly straight, meaning the ASDR grows by an almost constant factor per age step. In these data it doubles roughly every 7 to 8 years, the pattern Gompertz described in 1825. Lesson 1.7 builds mortality models on exactly this regularity.

Two working habits to take away: plot mortality schedules on a log scale by default, and check the hump separately for young males, because sex-specific external causes live there.


---

## Part 2: IMR and U5MR Are Probabilities, Not Rates

The **infant mortality rate** is the most cited number in all of population health:

$$
\text{IMR} = \frac{\text{deaths under age 1 in year } t}{\text{live births in year } t} \times 1000
$$

Three traps hide in this definition.

**Trap 1: it is not a rate.** In Carmichael's taxonomy (Ch. 1), a *rate* divides events in a period by the *average* population at risk, while a *probability* divides events during a life phase by the population *entering* the phase. Births are a flow of entrants into age 0, so the IMR is a probability ($q_0$), wearing the word "rate" by historical accident.

**Trap 2: the numerator belongs to two different years.** An infant death in year $t$ can be a baby born in $t$ or in $t-1$: on the Lexis diagram from [Lesson 1.1a](demographic-m1l1a-lexis-diagram), the square [age 0–1] × [year $t$] is split by a diagonal into triangles belonging to two birth cohorts. The IMR breaches Carmichael's *principle of correspondence* (numerator and denominator should cover the same people). The errors partly cancel, unless births are changing fast, as the computation below shows.

**Trap 3: why not just divide by the population aged 0?** Because exposure inside age 0 is violently uneven: deaths concentrate in the first days and weeks of life, so the evenly-spread assumption that justifies mid-year denominators fails badly here (Carmichael flags infant mortality as the standing exception). The count of births is measured better than person-years lived at age 0, so births win as the denominator.

The **under-five mortality rate** extends the same idea: $ {}_5q_0 $, the probability of dying before exact age 5 per 1,000 live births. Under-5 mortality is where development, nutrition, and infection bite hardest, and both IMR and U5MR can be estimated from survey birth histories where vital registration is incomplete, which is why DHS programs collect them. Now the numbers:


In [ ]:
# ---- 2. Carmichael's worked example: IMR vs the death rate at age 0 ----
# Ch. 1, p. 30: same numerator, two denominators, two concepts
births, infant_deaths, pop_age0 = 97_600, 9_760, 91_800   # births and mid-year pop aged 0, 2005

imr = infant_deaths / births * 1000
m0 = infant_deaths / pop_age0 * 1000
print(f'IMR              = {infant_deaths}/{births} x 1000 = {imr:.1f} per 1,000 live births')
print(f'death rate age 0 = {infant_deaths}/{pop_age0} x 1000 = {m0:.1f} per 1,000 aged 0')
print(f'Identity check: m0 / IMR = births / P(0) = {births / pop_age0:.4f} = {m0 / imr:.4f}')

# ---- 3. Trap 2 in action: the period mismatch when births are falling ----
# Split infant deaths in year t by birth cohort using a separation factor f:
# fraction f of age-0 deaths in year t come from births in year t, (1-f) from year t-1.
# (The rigorous version of f arrives in Lesson 1.5a.)
f, q0 = 0.6, 0.100                     # 60% from the current cohort; true risk 100 per 1,000
for label, g in [('births falling 5%/yr', 0.95), ('births flat', 1.0), ('births rising 5%/yr', 1.05)]:
    B_t = 97_600
    B_prev = B_t / g                   # last year's births
    D_t = q0 * (f * B_t + (1 - f) * B_prev)   # true deaths, both triangles
    imr_naive = D_t / B_t * 1000
    print(f'{label:22s}: naive IMR = {imr_naive:6.1f} vs true q0 = {q0*1000:.1f} '
          f'(bias {imr_naive / (q0 * 1000) - 1:+.1%})')

Two lessons from the numbers.

First, the IMR and the death rate at age 0 had the *same numerator* yet differed by 6.3%, purely because the denominators answer different questions: "out of every 1,000 babies born" versus "out of every 1,000 person-years lived at age 0." The identity $m_0 = \text{IMR} \times B/P(0)$ says they coincide only when births equal the mid-year population of infants.

Second, the period mismatch is real but bounded: with 5% annual changes in births, the naive IMR is off by about 2%, biased *upward* when births fall (the numerator contains babies from last year's larger cohort) and downward when births rise. This is why Wachter (§2.4) calls the IMR a pragmatic hybrid: it is computable from vital registration alone, with no census, and its built-in error stays modest unless fertility is moving fast.


---

## Part 3: MMR Is a Ratio, Not a Rate

The **maternal mortality ratio** completes the early-life indicator family:

$$
\text{MMR} = \frac{\text{maternal deaths in year } t}{\text{live births in year } t} \times 100{,}000
$$

A maternal death (WHO, ICD-10) is the death of a woman while pregnant or within 42 days of the end of pregnancy, from any cause related to or aggravated by the pregnancy or its management, excluding accidental and incidental causes.

Notice what the denominator is and is not. Live births stand in for the true population at risk, which is *pregnant women*, an unobservable stock in most statistical systems. So the MMR is a **ratio** in Carmichael's taxonomy: same units top and bottom would not even make sense. The multiplier is 100,000 rather than 1,000 because maternal death is mercifully rare in low-mortality settings, and per-1,000 values would be decimal dust.

The data-quality trap is worse than the definitional one. Maternal deaths are routinely misclassified on death certificates (a hemorrhage recorded without the pregnancy), and registration-based MMRs therefore understate the truth wherever cause-of-death coding is weak; in low-registration settings the number comes from survey methods such as the sisterhood approach instead. Treat any MMR as carrying a wide error band.


In [ ]:
# ---- 4. The indicator ladder, side by side ----
ladder = pd.DataFrame([
    ['CDR',   'all deaths / mid-year population',        'person-years (all ages)',  'true rate, crude', '1,000'],
    ['ASDR',  'deaths aged x..x+n / pop aged x..x+n',    'person-years (age band)',  'true rate, refined', '1,000'],
    ['IMR',   'deaths under 1 / live births',            'births (a cohort flow)',   'probability (q0)', '1,000'],
    ['U5MR',  'deaths under 5 / live births',            'births (a cohort flow)',   'probability (5q0)', '1,000'],
    ['MMR',   'maternal deaths / live births',           'births (proxy for pregnancies)', 'ratio', '100,000'],
], columns=['indicator', 'definition', 'denominator', 'measure type', 'reported per'])
print(ladder.to_string(index=False))

Keep the ladder in mind whenever an indicator crosses your desk. The first question is never "how big is it?" but "what type is it, and what exactly is in the denominator?"


---

## Part 4: Takeaways and Exercises

**Takeaways**

1. The ASDR ($ {}_nM_x $) is a true, refined rate. Plotted on a log scale it has a stable anatomy: high infancy, a minimum near ages 10–14, a young-adult accident hump, and a near-geometric senescent rise doubling every 7 to 8 years.
2. The IMR is a probability in a rate's clothing: births are entrants, not exposure. Its numerator spans two birth cohorts, so it is biased when births change fast, and the population aged 0 is a poor substitute denominator because exposure inside age 0 is concentrated in the first weeks.
3. U5MR ($ {}_5q_0 $) extends the infant idea to the ages where environment bites hardest; both it and IMR are estimable from survey birth histories.
4. The MMR is a ratio per 100,000 live births, with pregnancies as the unobservable true denominator and misclassification as the bigger practical problem.
5. Before comparing any two mortality numbers, classify them: count, true rate, probability, or ratio. Most public blunders are measure-type confusions.

**Exercises**

1. Using `asdr` from the first code cell, estimate the ASDR doubling time separately for Malaysian males and Australian males over ages 35–39 to 80–84 (use the sex-specific `rates` table). Does the Gompertz regularity hold equally well in both?
2. In 2005 a country records 100,000 births, a mid-year population aged 0 of 95,000, and 1,100 infant deaths. Compute the IMR and the death rate at age 0, and verify the identity $m_0 = \text{IMR} \times B/P(0)$.
3. If births fall 10% per year and the separation factor is 0.6, how large is the IMR bias? Repeat with $f = 0.8$ (mortality more concentrated in the first days) and explain the direction of the change.
4. Classify each by measure type (count, true rate, probability, ratio): the crude birth rate, the total fertility rate, the MMR, the prevalence of diabetes, the IMR. For each, name the denominator trap if it has one.
5. Why does the evenly-spread exposure assumption fail at age 0 but work acceptably at age 40? Sketch the within-year pattern of deaths for the two ages.

---

**Next post: Lesson 1.2, Direct Standardization.** Lesson 1.1b ended with an improvised fix, applying each country's schedule to the other's composition. Lesson 1.2 formalizes that fix, introduces the standard populations demographers actually use (Segi, WHO World Standard), and shows why only *ratios* of standardized rates are interpretable.
